In [3]:
import numpy as np
import pandas as pd
import sklearn as sk
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels as sm

def fa(n,m, a,b):
    f,a = plt.subplots(n,m, figsize = (a,b))
    return f,a

In [4]:
prices = pd.concat([pd.read_csv(f"prices_round_4_day_{i}.csv", delimiter = ';') for i in [1, 2, 3]])
trades = pd.concat([pd.read_csv(f"trades_round_4_day_{i}.csv", delimiter = ';') for i in [1, 2, 3]])
prices = {product: prices[prices['product'] == product] for product in prices['product'].unique()}
hp, ve = prices['HYDROGEL_PACK'], prices['VELVETFRUIT_EXTRACT']

In [9]:
hp.head()

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
1,1,0,HYDROGEL_PACK,9950,13,9947.0,23.0,NaN,NaN,9966,13,9968.0,23.0,NaN,NaN,9958.0,0.0
20,1,100,HYDROGEL_PACK,9953,15,9950.0,21.0,NaN,NaN,9969,15,9971.0,21.0,NaN,NaN,9961.0,0.0
33,1,200,HYDROGEL_PACK,9953,14,9951.0,21.0,NaN,NaN,9969,14,9972.0,21.0,NaN,NaN,9961.0,0.0
47,1,300,HYDROGEL_PACK,9952,15,9950.0,26.0,NaN,NaN,9968,15,9971.0,26.0,NaN,NaN,9960.0,0.0
49,1,400,HYDROGEL_PACK,9953,10,9951.0,26.0,NaN,NaN,9969,10,9972.0,26.0,NaN,NaN,9961.0,0.0


In [48]:
TICK_HORIZON = 50
TIMESTAMPS_PER_TICK = 100
TIMESTAMP_HORIZON = TICK_HORIZON * TIMESTAMPS_PER_TICK
MIN_TRADES = 30

import pandas as pd
import numpy as np

def compute_informed_traders(price_df: pd.DataFrame,
                             trades_df: pd.DataFrame,
                             product: str,
                             fair_value_fn):
    
    # --- 1. Prep ---
    price_df = price_df.sort_values("timestamp").copy()
    trades_df = trades_df[trades_df["symbol"] == product].copy()
    
    # --- 2. Fair value ---
    price_df["FV"] = fair_value_fn(price_df)
    price_df["FV_future"] = price_df["FV"].shift(-TICK_HORIZON)
    price_df["dFV"] = price_df["FV_future"] - price_df["FV"]
    
    price_small = price_df[["timestamp", "FV", "FV_future", "dFV", "mid_price"]]
    
    # --- 3. Merge ---
    trades_df = trades_df.merge(price_small, on="timestamp", how="left")
    trades_df = trades_df.dropna(subset=["FV", "FV_future", "dFV"])
    
    # --- 4. Expand trades ---
    buyer = trades_df.copy()
    buyer["trader"] = buyer["buyer"]
    buyer["sign"] = 1
    
    seller = trades_df.copy()
    seller["trader"] = seller["seller"]
    seller["sign"] = -1
    
    all_trades = pd.concat([buyer, seller], ignore_index=True)
    
    # --- 5. Core metrics ---
    
    # correctness
    all_trades["correct"] = (all_trades["sign"] * all_trades["dFV"]) > 0
    
    # FULL edge (same as before)
    all_trades["edge"] = all_trades["sign"] * (all_trades["FV_future"] - all_trades["price"])
    
    # --- NEW DECOMPOSITION ---
    
    # entry edge: did they trade at a good price vs CURRENT FV
    all_trades["entry_edge"] = all_trades["sign"] * (all_trades["FV"] - all_trades["price"])
    
    # timing edge: did FV move in their favour AFTER trade
    all_trades["timing_edge"] = all_trades["sign"] * (all_trades["FV_future"] - all_trades["FV"])
    
    # signed trade for correlation
    all_trades["signed_trade"] = all_trades["sign"]
    
    # --- 6. Aggregation ---
    summary = all_trades.groupby("trader").agg(
        trades=("edge", "count"),
        
        # basic
        hit_rate=("correct", "mean"),
        
        # total edge
        avg_edge=("edge", "mean"),
        edge_std=("edge", "std"),
        
        # entry metrics
        avg_entry_edge=("entry_edge", "mean"),
        entry_edge_sd=("entry_edge", "std"),
        
        # timing metrics
        avg_timing_edge=("timing_edge", "mean"),
        timing_edge_sd=("timing_edge", "std"),
    )
    
    # --- 7. Info ratios (safe) ---
    summary["info_ratio"] = summary["avg_edge"] / summary["edge_std"].replace(0, np.nan)
    summary["entry_info_ratio"] = summary["avg_entry_edge"] / summary["entry_edge_sd"].replace(0, np.nan)
    summary["timing_info_ratio"] = summary["avg_timing_edge"] / summary["timing_edge_sd"].replace(0, np.nan)
    
    # --- 8. Safe correlation ---
    def trader_corr(x):
        if len(x) < MIN_TRADES:
            return np.nan
        
        s = x["signed_trade"].to_numpy()
        d = x["dFV"].to_numpy()
        
        mask = ~np.isnan(s) & ~np.isnan(d)
        s, d = s[mask], d[mask]
        
        if len(s) < MIN_TRADES:
            return np.nan
        
        s_std = np.std(s)
        d_std = np.std(d)
        
        if s_std == 0 or d_std == 0:
            return np.nan
        
        cov = np.mean((s - s.mean()) * (d - d.mean()))
        return cov / (s_std * d_std)
    
    summary["corr_future"] = all_trades.groupby("trader").apply(trader_corr)
    
    # --- 9. Filter ---
    summary = summary[summary["trades"] >= MIN_TRADES]
    
    # --- 10. Rank ---
    summary = summary.sort_values(
        ["avg_edge", "avg_timing_edge", "corr_future"],
        ascending=False
    )
    
    return summary, all_trades

In [49]:
def fair_value_velvet(df: pd.DataFrame) -> pd.Series:
    """
    Same as in r3-final-14.py but modified for use with a df.
    """

    # Extract columns
    bid_p3 = df["bid_price_3"]
    bid_v3 = df["bid_volume_3"]
    
    bid_p2 = df["bid_price_2"]
    bid_v2 = df["bid_volume_2"]
    
    ask_p3 = df["ask_price_3"]
    ask_v3 = df["ask_volume_3"]
    
    ask_p2 = df["ask_price_2"]
    ask_v2 = df["ask_volume_2"]

    # --- Valid masks (only include if BOTH price and volume exist) ---
    bid3_valid = bid_p3.notna() & bid_v3.notna()
    bid2_valid = bid_p2.notna() & bid_v2.notna()
    
    ask3_valid = ask_p3.notna() & ask_v3.notna()
    ask2_valid = ask_p2.notna() & ask_v2.notna()

    # --- Total value (only include valid entries) ---
    total = (
        (bid_p3 * bid_v3).where(bid3_valid, 0) +
        (bid_p2 * bid_v2).where(bid2_valid, 0) +
        (ask_p3 * ask_v3).where(ask3_valid, 0) +
        (ask_p2 * ask_v2).where(ask2_valid, 0)
    )

    # --- Total volume ---
    count = (
        bid_v3.where(bid3_valid, 0) +
        bid_v2.where(bid2_valid, 0) +
        ask_v3.where(ask3_valid, 0) +
        ask_v2.where(ask2_valid, 0)
    )

    fv = total / count

    fv[count == 0] = np.nan

    return fv

def fair_value_hydro(df: pd.DataFrame) -> pd.Series:
    """
    Exact equivalent of compute_fair_value but operating on a DataFrame.
    Preserves prev_makers state across rows (timestamps).
    """

    df = df.sort_values("timestamp").copy()
    
    fv = []
    prev_makers: Dict = {}

    for _, row in df.iterrows():
        
        # --- Extract levels (NaN-safe) ---
        def get(p, v, sign=1):
            if pd.notna(p) and pd.notna(v):
                return p, sign * v
            return None, None

        bid_price_1, bid_volume_1 = get(row["bid_price_1"], row["bid_volume_1"])
        bid_price_2, bid_volume_2 = get(row["bid_price_2"], row["bid_volume_2"])
        bid_price_3, bid_volume_3 = get(row["bid_price_3"], row["bid_volume_3"])

        ask_price_1, ask_volume_1 = get(row["ask_price_1"], row["ask_volume_1"], sign=-1)
        ask_price_2, ask_volume_2 = get(row["ask_price_2"], row["ask_volume_2"], sign=-1)
        ask_price_3, ask_volume_3 = get(row["ask_price_3"], row["ask_volume_3"], sign=-1)

        # --- initialise filtered makers ---
        f_mbp1 = f_mbv1 = f_mbp2 = f_mbv2 = None
        f_map1 = f_mav1 = f_map2 = f_mav2 = None

        # --- BID SIDE (exact logic) ---
        if bid_price_3 is not None:
            f_mbp1, f_mbv1 = bid_price_2, bid_volume_2
            f_mbp2, f_mbv2 = bid_price_3, bid_volume_3

        elif bid_price_2 is not None:
            if bid_volume_1 is not None and (bid_volume_1 < 10 or abs(bid_price_1 - bid_price_2) >= 5):
                if bid_volume_2 >= 20:
                    f_mbp2, f_mbv2 = bid_price_2, bid_volume_2
                else:
                    f_mbp1, f_mbv1 = bid_price_2, bid_volume_2
            else:
                f_mbp1, f_mbv1 = bid_price_1, bid_volume_1
                f_mbp2, f_mbv2 = bid_price_2, bid_volume_2

        elif bid_price_1 is not None:
            if bid_volume_1 >= 20:
                f_mbp2, f_mbv2 = bid_price_1, bid_volume_1
            elif bid_volume_1 >= 10:
                f_mbp1, f_mbv1 = bid_price_1, bid_volume_1

        # --- ASK SIDE (exact logic) ---
        if ask_price_3 is not None:
            f_map1, f_mav1 = ask_price_2, ask_volume_2
            f_map2, f_mav2 = ask_price_3, ask_volume_3

        elif ask_price_2 is not None:
            if ask_volume_1 is not None and (ask_volume_1 < 10 or abs(ask_price_1 - ask_price_2) >= 5):
                if ask_volume_2 >= 20:
                    f_map2, f_mav2 = ask_price_2, ask_volume_2
                else:
                    f_map1, f_mav1 = ask_price_2, ask_volume_2
            else:
                f_map1, f_mav1 = ask_price_1, ask_volume_1
                f_map2, f_mav2 = ask_price_2, ask_volume_2

        elif ask_price_1 is not None:
            if ask_volume_1 >= 20:
                f_map2, f_mav2 = ask_price_1, ask_volume_1
            elif ask_volume_1 >= 10:
                f_map1, f_mav1 = ask_price_1, ask_volume_1

        # --- Persist memory ---
        new_prev = dict(prev_makers)

        if f_mbp1 is not None:
            new_prev["maker_bid_price_1"]  = f_mbp1
            new_prev["maker_bid_volume_1"] = f_mbv1
        if f_mbp2 is not None:
            new_prev["maker_bid_price_2"]  = f_mbp2
            new_prev["maker_bid_volume_2"] = f_mbv2
        if f_map1 is not None:
            new_prev["maker_ask_price_1"]  = f_map1
            new_prev["maker_ask_volume_1"] = f_mav1
        if f_map2 is not None:
            new_prev["maker_ask_price_2"]  = f_map2
            new_prev["maker_ask_volume_2"] = f_mav2

        # --- Retrieve ---
        mbp1 = new_prev.get("maker_bid_price_1")
        mbp2 = new_prev.get("maker_bid_price_2")
        map1 = new_prev.get("maker_ask_price_1")
        map2 = new_prev.get("maker_ask_price_2")

        # --- Fair value ---
        if mbp1 is not None and mbp2 is not None and map1 is not None and map2 is not None:
            fair = (mbp1 + mbp2 + map1 + map2) / 4
        elif mbp1 is not None and map1 is not None:
            fair = (mbp1 + map1) / 2
        elif mbp2 is not None and map2 is not None:
            fair = (mbp2 + map2) / 2
        else:
            fair = np.nan

        fv.append(fair)
        prev_makers = new_prev  # update state

    return pd.Series(fv, index=df.index)

ve_summary, ve_trades = compute_informed_traders(
    ve, trades, "VELVETFRUIT_EXTRACT", fair_value_velvet
)

hp_summary, hp_trades = compute_informed_traders(
    hp, trades, "HYDROGEL_PACK", fair_value_hydro
)

In [50]:
ve_summary.head()

,trades,hit_rate,avg_edge,edge_std,avg_entry_edge,entry_edge_sd,avg_timing_edge,timing_edge_sd,info_ratio,entry_info_ratio,timing_info_ratio,corr_future
trader,,,,,,,,,,,,
Mark 14,1429,0.496851,2.270708,20.464467,1.824539,19.345792,0.446169,20.666301,0.110959,0.094312,0.021589,0.021138
Mark 01,914,0.474836,2.059192,20.229133,2.873821,20.341649,-0.814629,18.798813,0.101793,0.141278,-0.043334,-0.043544
Mark 22,268,0.518657,0.836420,21.322408,0.220029,19.796800,0.616391,20.564647,0.039227,0.011114,0.029973,0.042974
Mark 67,355,0.518310,0.663809,19.991544,0.581088,18.679183,0.082722,19.959800,0.033205,0.031109,0.004144,NaN
Mark 49,265,0.456604,-0.769155,17.335650,-0.814358,18.108756,0.045202,20.004234,-0.044368,-0.044970,0.002260,0.015112


In [51]:
hp_summary.head()

,trades,hit_rate,avg_edge,edge_std,avg_entry_edge,entry_edge_sd,avg_timing_edge,timing_edge_sd,info_ratio,entry_info_ratio,timing_info_ratio,corr_future
trader,,,,,,,,,,,,
Mark 14,3009,0.495846,7.000000,39.217391,7.528332,39.036966,-0.528332,33.063874,0.178492,0.192851,-0.015979,-0.016210
Mark 22,57,0.491228,-4.750000,38.392911,-4.105263,40.455662,-0.644737,29.019888,-0.123721,-0.101476,-0.022217,-0.026186
Mark 38,3066,0.499348,-6.781556,39.228212,-7.312052,39.088594,0.530496,32.989031,-0.172874,-0.187064,0.016081,0.016241
